In [ ]:
# Example 1: top-level package surface inspection.
import turboquant

public_names = [n for n in dir(turboquant) if not n.startswith('_')]
print(f"turboquant public attributes: {len(public_names)}")
for name in public_names[:25]:
    obj = getattr(turboquant, name)
    print(f"  {name}: {type(obj).__name__}")

assert turboquant is not None

In [ ]:
# Example 2: dependency stack importability (torch / transformers / scipy / numpy).
import numpy as np
import scipy
import torch
import transformers

print(f"numpy:        {np.__version__}")
print(f"scipy:        {scipy.__version__}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")

# turboquant is a quantization library targeting torch / transformers — verify those load.
assert hasattr(torch, 'nn')
assert hasattr(transformers, 'AutoModel')

In [ ]:
# Example 3: enumerate turboquant submodules and pick out callables/classes.
import importlib
import inspect
import pkgutil

import turboquant

submodule_count = 0
class_count = 0
func_count = 0
for info in pkgutil.walk_packages(turboquant.__path__, prefix='turboquant.'):
    submodule_count += 1
    try:
        mod = importlib.import_module(info.name)
    except Exception:
        continue
    for _, obj in inspect.getmembers(mod):
        if inspect.isclass(obj) and obj.__module__.startswith('turboquant'):
            class_count += 1
        elif inspect.isfunction(obj) and obj.__module__.startswith('turboquant'):
            func_count += 1

print(f"submodules walked: {submodule_count}")
print(f"turboquant-owned classes:   {class_count}")
print(f"turboquant-owned functions: {func_count}")

In [ ]:
# Example 4: exercise a tiny torch tensor op — sanity-checks the backing stack turboquant rides.
import torch

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
z = x @ y

expected = torch.tensor([[19.0, 22.0], [43.0, 50.0]])
print(f"x @ y =\n{z}")
assert torch.allclose(z, expected)

# Quantization is turboquant's domain — verify torch's int8 quantize op is reachable.
q = torch.quantize_per_tensor(x, scale=0.1, zero_point=0, dtype=torch.qint8)
print(f"quantized dtype: {q.dtype}, scale: {q.q_scale()}")
assert q.dtype == torch.qint8

In [ ]:
# Example 5: package metadata + final pass signal.
import importlib.metadata

import turboquant

try:
    dist = importlib.metadata.distribution('turboquant')
    print(f"name:    {dist.metadata['Name']}")
    print(f"version: {dist.version}")
    summary = dist.metadata.get('Summary', '')
    if summary:
        print(f"summary: {summary}")
    requires = dist.requires or []
    print(f"requires ({len(requires)}):")
    for r in requires[:10]:
        print(f"  {r}")
    assert dist.metadata['Name'].lower() == 'turboquant'
except importlib.metadata.PackageNotFoundError:
    print("distribution metadata not found; falling back to module check")
    assert turboquant is not None

print("\nturboquant smoke test: PASS")